# Демо 2. Шаблони розмови і підготовка даних для SFT

В демо 1 донавчання моделі було пов'язане з заміною голови, а отже частковою зміною архітектури нейронів. В багатьох випадках при донавчанні LLM моделей архітектура не змінюється, а змінюється радше формат введених і виведених данних. Для цього потрібно зрозуміти, **у якому вигляді модель бачить діалог**.
В будь-якому випадку фізично на вхід моделі подається тензор з токенами, а за те як з цих токенів утворюються семантичні повідомлення відповідають **шаблони розмови** (chat templates).
Під час донавчання модель може вивчити новий шаблон розмови.

Донавчання LLM методом SFT виконується на діалогах, відформатованих шаблоном розмови.
У цьому ноутбуці розглянемо, що таке шаблон розмови і як підготувати дані у форматі діалогів;
саме донавчання виконаємо в демо 3.

План:
1. Завантажити base- та instruct-версії SmolLM3.
2. Подивитися, як шаблон розмови перетворює різні типи діалогів на текст.
3. Порівняти відповіді base- та instruct-моделі.
4. Перевірити режими міркування `/think` і `/no_think`.
5. Порівняти шаблони різних моделей.
6. Підготувати датасет для SFT у форматі `messages`.

Демо за мотивами вправ курсу [smol-course](https://huggingface.co/learn/smol-course) і розділу 11
[Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter11/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
import time
from IPython import get_ipython
_cell_times, _t_start = [], time.time()
def _pre_cell(info):
    globals()["_t_cell"] = time.time()
def _post_cell(result):
    lines = [l for l in (result.info.raw_cell or "").splitlines() if l.strip() and not l.lstrip().startswith("#")]
    _cell_times.append((lines[0][:80] if lines else "", round(time.time() - globals().get("_t_cell", time.time()), 1)))
get_ipython().events.register("pre_run_cell", _pre_cell)
get_ipython().events.register("post_run_cell", _post_cell)

In [2]:
!pip install -q "transformers>=4.55.0" "datasets" "accelerate"

In [3]:
import warnings

import torch
from datasets import get_dataset_split_names, load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Пам'ять GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Apple MPS")
else:
    device = "cpu"
    warnings.warn("Обчислення на CPU: генерація буде дуже повільною, потрібна GPU")

GPU: Tesla T4
Пам'ять GPU: 15.6 GB


In [4]:
if device == "cuda":
    !nvidia-smi

Wed Nov 18 07:24:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Завантаження моделей

[**SmolLM3**](https://huggingface.co/blog/smollm3) - відкрита модель від Hugging Face на 3 млрд параметрів. Вона має дві версії:
- **Base** - лише після попереднього тренування: продовжує текст, але не веде діалог;
- **Instruct** (`SmolLM3-3B`) - донавчена виконувати інструкції й вести діалог.

In [5]:
base_model_name = "HuggingFaceTB/SmolLM3-3B-Base"
instruct_model_name = "HuggingFaceTB/SmolLM3-3B"

base_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
instruct_tokenizer = AutoTokenizer.from_pretrained(instruct_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, dtype=torch.float16, device_map="auto"
)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_name, dtype=torch.float16, device_map="auto"
)
print("Моделі завантажено")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (12 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Моделі завантажено


## 2. Як шаблон розмови форматує діалог

Діалог задається списком повідомлень із ролями:
- `system` - інструкції для моделі (роль, стиль, обмеження);
- `user` - повідомлення користувача;
- `assistant` - відповіді моделі.

Метод `apply_chat_template` перетворює цей список на рядок зі спеціальними токенами.

Параметр `add_generation_prompt`:
- `False` - для передачі в модель завершених діалогів, тобто для **тренувальних даних**;
- `True` - додає в кінець заголовок відповіді асистента, щоб модель почала відповідати; використовується для **інференсу**.

Для SmolLM3 у системне повідомлення можна додати прапорець `/think` або `/no_think`,
який вмикає чи вимикає режим міркування.

In [6]:
conversations = {
    "simple_qa": [
        {"role": "system", "content": "/no_think"},
        {"role": "user", "content": "What is machine learning?"},
    ],
    "with_system": [
        {
            "role": "system",
            "content": "You are a helpful AI assistant specialized in explaining technical concepts clearly. /no_think",
        },
        {"role": "user", "content": "What is machine learning?"},
    ],
    "multi_turn": [
        {"role": "system", "content": "You are a math tutor. /no_think"},
        {"role": "user", "content": "What is calculus?"},
        {
            "role": "assistant",
            "content": "Calculus is a branch of mathematics that deals with rates of change and accumulation of quantities.",
        },
        {"role": "user", "content": "Can you give me a simple example?"},
    ],
    "reasoning_task": [
        {"role": "system", "content": "/think"},
        {
            "role": "user",
            "content": "Solve step by step: If a train travels 120 miles in 2 hours, what is its average speed?",
        },
    ],
}

for conv_type, messages in conversations.items():
    print(f"--- {conv_type.upper()} ---")

    formatted_complete = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    formatted_prompt = instruct_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    print("Завершений діалог (add_generation_prompt=False):")
    print(formatted_complete)
    print("\nДля інференсу (add_generation_prompt=True):")
    print(formatted_prompt)
    print("\n" + "=" * 50 + "\n")

--- SIMPLE_QA ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>


Для інференсу (add_generation_prompt=True):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face.

<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
<think>

</think>



--- WITH_SYSTEM ---
Завершений діалог (add_generation_prompt=False):
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /no_think

## Custom Instructions

You are a helpful AI assistant specialized in explaining technical concepts clearly.

<|im_s

**Що варто помітити:**
- Шаблон SmolLM3 сам додає до системного повідомлення блок метаданих (дата відсічення знань, поточна дата, режим міркування)
  і стандартну інструкцію, якщо власної немає. Модель бачить більше, ніж ми написали.
- Кожне повідомлення обгорнуте токенами `<|im_start|>роль` … `<|im_end|>` (формат ChatML).
- З `add_generation_prompt=True` рядок закінчується на `<|im_start|>assistant`: з цього місця модель починає генерувати.

## 3. Base проти Instruct

Ставимо обом моделям одне питання. Base-модель отримує сирий текст без шаблону,
instruct-модель - діалог, відформатований шаблоном розмови.

In [7]:
def generate(model, tokenizer, text, max_new_tokens=150, temperature=0.7):
    """Генерує продовження тексту й повертає лише нові токени, без промпту."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [8]:
test_prompt = "Explain me, what is quantum computing?"

print("🤖 BASE-МОДЕЛЬ:")
print("-" * 50)
print(generate(base_model, base_tokenizer, test_prompt))

instruct_messages = [
    {"role": "system", "content": "/no_think"},
    {"role": "user", "content": test_prompt},
]
instruct_formatted = instruct_tokenizer.apply_chat_template(
    instruct_messages, tokenize=False, add_generation_prompt=True
)
print("\n" + "=" * 50)
print("💬 INSTRUCT-МОДЕЛЬ:")
print("-" * 50)
print(generate(instruct_model, instruct_tokenizer, instruct_formatted))

🤖 BASE-МОДЕЛЬ:
--------------------------------------------------


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 This is the question I’ve always wanted to know the answer to. Quantum computing is another new concept that is gaining ground in the field of computer science. It is a concept that is still being developed and is still in its early stages of development. In this article, I’ll try to explain quantum computing and the various concepts involved in it. Quantum computing is a new concept that is being developed by a number of researchers in the field. One of the main concepts in quantum computing is the idea of quantum information. Quantum information is a type of information that can be used to process information. It is also known as quantum data. Quantum information is a collection of data that has a certain structure and is organized in a certain way. Quantum information can be

💬 INSTRUCT-МОДЕЛЬ:
--------------------------------------------------
Quantum computing is a revolutionary field of computing that leverages the principles of quantum mechanics to process and analyze informati

## 4. Режими міркування `/think` і `/no_think`

SmolLM3 - гібридна модель: у режимі `/think` вона спочатку генерує міркування в блоці `<think>…</think>`,
а потім відповідь; у режимі `/no_think` блок міркувань порожній.

In [9]:
reasoning_prompts = [
    "What is 15 × 24? Show your work.",
    "A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?",
    "If I have $50 and spend $18.75 on lunch and $12.30 on a book, how much money do I have left?",
]

for thinking_flag in ["/no_think", "/think"]:
    print(f"===== Режим: {thinking_flag} =====\n")
    for i, prompt in enumerate(reasoning_prompts, 1):
        messages = [
            {"role": "system", "content": thinking_flag},
            {"role": "user", "content": prompt},
        ]
        formatted = instruct_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        answer = generate(instruct_model, instruct_tokenizer, formatted, max_new_tokens=300, temperature=0.3)
        print(f"Задача {i}: {prompt}")
        print(f"Відповідь: {answer}")
        print("\n" + "-" * 50 + "\n")

===== Режим: /no_think =====

Задача 1: What is 15 × 24? Show your work.
Відповідь: To find the product of 15 and 24, we can use the standard multiplication algorithm. Here's how to do it step by step:

```
   15
× 24
------
```

First, we multiply 15 by the units digit of 24, which is 4:

```
   15
×  4
------
  60
```

Next, we multiply 15 by the tens digit of 24, which is 2. Since this is the tens place, we need to shift the result one position to the left:

```
   15
×  2
------
  30
```

Now, we add the two partial products:

```
   15
× 24
------
  60
+ 30
------
  360
```

Therefore, the product of 15 and 24 is 360.

Alternatively, we can also use the distributive property to break down the multiplication:

15 × 24 = 15 × (20 + 4) = 15 × 20 + 15 × 4 = 300 + 60 = 360.

Both methods yield the same result.

--------------------------------------------------

Задача 2: A recipe calls for 2 cups of flour for 12 cookies. How much flour is needed for 30 cookies?
Відповідь: To determine

## 5. У кожної моделі свій шаблон

Ролі `system`, `user`, `assistant` - це домовленість на рівні формату `messages`.
У токенах кожна модель записує їх по-своєму. Порівняємо шаблони кількох моделей.

In [10]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi!"},
    {"role": "assistant", "content": "Hello! How can I help?"},
]

for name in ["HuggingFaceTB/SmolLM3-3B", "Qwen/Qwen3-0.6B", "microsoft/Phi-3.5-mini-instruct"]:
    tok = AutoTokenizer.from_pretrained(name)
    print(f"===== {name} =====")
    print(tok.apply_chat_template(messages, tokenize=False))
    print()

===== HuggingFaceTB/SmolLM3-3B =====
<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful assistant.

<|im_start|>user
Hi!<|im_end|>
<|im_start|>assistant
Hello! How can I help?<|im_end|>




config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (7 outputs left out)

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

===== microsoft/Phi-3.5-mini-instruct =====
<|system|>
You are a helpful assistant.<|end|>
<|user|>
Hi!<|end|>
<|assistant|>
Hello! How can I help?<|end|>
<|endoftext|>



Прапорці режиму міркування також залежать від моделі. У Qwen3, наприклад,
режим вмикається параметром `enable_thinking` у `apply_chat_template`.

In [11]:
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
question = [{"role": "user", "content": "What is 2 + 2?"}]

for enable_thinking in [True, False]:
    print(f"===== enable_thinking={enable_thinking} =====")
    print(qwen_tokenizer.apply_chat_template(
        question, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    ))

===== enable_thinking=True =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant

===== enable_thinking=False =====
<|im_start|>user
What is 2 + 2?<|im_end|>
<|im_start|>assistant
<think>

</think>




### А що з base-моделлю?

Base-модель не донавчали на діалогах, тому шаблон у її токенізаторі або відсутній, або модель ніколи не бачила його токенів у ролі розмітки діалогу.
Перед SFT шаблон для base-моделі задають самостійно - найпростіше взяти його з instruct-версії тієї самої моделі.

In [12]:
print("Шаблон є в base-токенізаторі:  ", base_tokenizer.chat_template is not None)
print("Шаблон є в instruct-токенізаторі:", instruct_tokenizer.chat_template is not None)

Шаблон є в base-токенізаторі:   False
Шаблон є в instruct-токенізаторі: True


## 6. Підготовка датасету для SFT

Для SFT дані зберігають у форматі `messages` - тобто у списку повідомлень із ролями.
Бібліотека TRL (`SFTTrainer`) сама застосовує до нього шаблон розмови під час тренування.

### SmolTalk2

**SmolTalk2** - датасет, на якому донавчали SmolLM3. Він складається з багатьох частин (splits) з різних джерел.
Список частин можна отримати без завантаження самих даних.

In [13]:
splits = get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")
print(f"Кількість частин: {len(splits)}")
for s in splits:
    print(" -", s)

README.md:   0%|          | 0.00/23.5k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Кількість частин: 25
 - LongAlign_64k_Qwen3_32B_yarn_131k_think
 - OpenThoughts3_1.2M_think
 - aya_dataset_Qwen3_32B_think
 - multi_turn_reasoning_if_think
 - s1k_1.1_think
 - smolagents_toolcalling_traces_think
 - smoltalk_multilingual8_Qwen3_32B_think
 - smoltalk_systemchats_Qwen3_32B_think
 - table_gpt_Qwen3_32B_think
 - LongAlign_64k_context_lang_annotated_lang_6_no_think
 - Mixture_of_Thoughts_science_no_think
 - OpenHermes_2.5_no_think
 - OpenThoughts3_1.2M_no_think_no_think
 - hermes_function_calling_v1_no_think
 - smoltalk_multilingual_8languages_lang_5_no_think
 - smoltalk_smollm3_everyday_conversations_no_think
 - smoltalk_smollm3_explore_instruct_rewriting_no_think
 - smoltalk_smollm3_smol_magpie_ultra_no_think
 - smoltalk_smollm3_smol_rewrite_no_think
 - smoltalk_smollm3_smol_summarize_no_think
 - smoltalk_smollm3_systemchats_30k_no_think
 - table_gpt_no_think
 - tulu_3_sft_personas_instruction_following_no_think
 - xlam_traces_no_think
 - smoltalk_everyday_convs_reasoning_

In [14]:
split = "smoltalk_everyday_convs_reasoning_Qwen3_32B_think"
everyday = load_dataset("HuggingFaceTB/smoltalk2", data_files=f"SFT/{split}-*.parquet", split="train")
print(everyday)
print()
for message in everyday[0]["messages"]:
    print(f"[{message['role']}] {message['content'][:200]}")

SFT/smoltalk_everyday_convs_reasoning_Qw(…): reconstructing file:   0%|          |  0.00B / 6.33MB            

SFT/smoltalk_everyday_convs_reasoning_Qw(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2057
})

[user] Hi there
[assistant] <think>
Okay, the user sent "Hi there". That's a friendly greeting. I should respond in a welcoming way. Let me check the guidelines. I need to be helpful, keep the conversation going, and maybe ask h
[user] I'm looking for a healthy breakfast idea. What's a good option?
[assistant] <think>
Okay, the user is asking for a healthy breakfast idea. Let me think about what makes a breakfast healthy. It should be balanced, providing a mix of nutrients like protein, fiber, healthy fats,


Ця частина зібрана для режиму `/think`, тому відповіді асистента містять блок міркувань `<think>…</think>`.

Як видно з назви частини, відповіді в ній згенерувала модель Qwen3-32B. Такі **синтетичні дані** від сильнішої моделі -
поширений спосіб отримати тисячі прикладів для SFT без ручного написання відповідей. Помилки моделі-генератора при цьому переходять у дані,
тому синтетичні дані фільтрують.

### Перетворення власного датасету у формат `messages`

Більшість датасетів має інший формат, наприклад колонки "питання" і "відповідь".
Перетворимо датасет задач з математики **GSM8K** у формат `messages`.

In [15]:
gsm8k = load_dataset("openai/gsm8k", "main", split="train[:100]")
print("Оригінальний приклад:", gsm8k[0])


def to_messages(examples):
    """Перетворює пари 'питання - відповідь' на діалоги з ролями."""
    processed = []
    for question, answer in zip(examples["question"], examples["answer"]):
        processed.append([
            {"role": "system", "content": "You are a math tutor. Solve problems step by step."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ])
    return {"messages": processed}


gsm8k_messages = gsm8k.map(to_messages, batched=True, remove_columns=gsm8k.column_names)
print("\nУ форматі messages:", gsm8k_messages[0])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Оригінальний приклад: {'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'answer': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72'}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]


У форматі messages: {'messages': [{'content': 'You are a math tutor. Solve problems step by step.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}, {'content': 'Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.\n#### 72', 'role': 'assistant'}]}


### Що побачить модель під час тренування

Застосуємо шаблон розмови до підготовленого прикладу. Саме цей рядок модель отримає під час SFT.

In [16]:
text = instruct_tokenizer.apply_chat_template(
    gsm8k_messages[0]["messages"],
    tokenize=False,
    add_generation_prompt=False,
)
print(text)
print(f"Довжина в токенах: {len(instruct_tokenizer(text)['input_ids'])}")

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 18 November 2026
Reasoning Mode: /think

## Custom Instructions

You are a math tutor. Solve problems step by step.

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>

Довжина в токенах: 145


**Що варто помітити:** шаблон знову додав метадані й режим `/think`, хоча в даних їх немає,
а відповідь асистента не містить блоку міркувань. Якщо донавчати на таких прикладах, модель вчитиметься
відповідати без міркувань навіть у режимі `/think`. Тому формат даних і шаблон мають узгоджуватися з тим,
як модель використовуватиметься після донавчання.

### Тривалість виконання клітинок

In [17]:
import pandas as pd
print(f"Увесь ноутбук: {(time.time() - _t_start) / 60:.1f} хв")
pd.DataFrame(_cell_times, columns=["клітинка (перший рядок коду)", "секунд"]).sort_values("секунд", ascending=False).head(10)

Увесь ноутбук: 4.8 хв


,клітинка (перший рядок коду),секунд
4,"base_model_name = ""HuggingFaceTB/SmolLM3-3B-Base""",168.8
8,reasoning_prompts = [,61.4
2,import warnings,20.8
7,"test_prompt = ""Explain me, what is quantum com...",13.5
1,"!pip install -q ""transformers>=4.55.0"" ""datase...",8.9
9,messages = [,6.0
14,"gsm8k = load_dataset(""openai/gsm8k"", ""main"", s...",2.6
13,"split = ""smoltalk_everyday_convs_reasoning_Qwe...",1.6
12,"splits = get_dataset_split_names(""HuggingFaceT...",1.5
10,qwen_tokenizer = AutoTokenizer.from_pretrained...,0.8


---
# Відповіді на питання 0.1 (11-21)

**11. Як довго виконувалися клітинки ноутбуку `02_chat_templates`? Чи були проблеми з запуском, які?**

Увесь ноутбук на Colab T4 виконався за **≈ 4,8 хв**:

| що | час |
|---|---|
| завантаження SmolLM3-3B-Base і SmolLM3-3B (по ≈ 6 ГБ ваг кожна) | 169 с |
| генерація 6 відповідей у режимах `/no_think` і `/think` (секція 4) | 61 с |
| імпорт бібліотек | 21 с |
| порівняння Base / Instruct (секція 3) | 14 с |
| `pip install` | 9 с |
| токенізатори інших моделей (секція 5) | 6 с |
| GSM8K, SmolTalk2 | 2-3 с |

Найдовше займає **завантаження моделей**: 2 × 3 млрд параметрів, на T4 вони займають ≈ 12 ГБ з 15. Генерація теж повільна,
бо `generate` видає токени по одному, а в режимі `/think` кожна відповідь - 300 токенів.

**Проблем із запуском не було**, помилок немає. Були лише попередження: про запити до HF Hub без токена, про
`clean_up_tokenization_spaces` для BPE-токенізатора і про поле `rope_parameters` у конфігурації Phi-3.5. На результат вони не впливають.

**12. Для чого змінювати параметр `add_generation_prompt` в методі `apply_chat_template()` при використанні моделі? Як по згенерованим розмовам можна дізнатися значення цього параметру, яке було задано при виклику метода?**

- `add_generation_prompt=True` - для **інференсу**: у кінець рядка додається заголовок нової репліки асистента
  (`<|im_start|>assistant`). Модель - це "продовжувач тексту"; без заголовка вона не знає, що зараз її черга,
  і може продовжити репліку користувача, згенерувати нову репліку `user` або одразу поставити `<|im_end|>`.
- `add_generation_prompt=False` - для **тренувальних даних**: діалог уже завершений, відповідь асистента є,
  і зайвий порожній заголовок у кінці лише зіпсував би приклад.

Як дізнатися по виводу: дивимось на **кінець рядка**. Якщо він закінчується на `<|im_start|>assistant` (+ перенесення рядка;
для Qwen3 з вимкненим міркуванням - ще й порожній блок `<think></think>`) - було `True`. Якщо закінчується на `<|im_end|>`
після останнього повідомлення - було `False`. У виводі секції 2 обидва варіанти надруковані поруч, і вони відрізняються лише цим хвостом.

**13. Чим відрізняються ролі `system` від `assistant` в шаблонах розмов?**

- `system` - інструкції від **розробника**: хто модель, у якому стилі й мовою відповідати, які обмеження, режим `/think`/`/no_think`.
  Задається один раз на початку, користувач його зазвичай не бачить, а модель навчена ставити ці інструкції вище за прохання користувача.
- `assistant` - репліки **самої моделі**. В історії діалогу це її попередні відповіді (контекст для наступної), у тренувальних
  даних - **цільовий текст**: саме на токенах `assistant` рахується loss під час SFT, тобто модель вчиться генерувати їх.

Коротко: `system` каже моделі, *як* відповідати, `assistant` - це *що* вона відповідає.

**14. Прокоментуйте вивід даних в секції 2 "Як шаблон розмови форматує діалог". Які секції було додано шаблонізатором розмов і для чого?**

Для всіх чотирьох діалогів шаблон SmolLM3 перетворив список повідомлень на текст і **додав те, чого ми не писали**:
- **спецтокени ChatML** `<|im_start|>роль` … `<|im_end|>`: модель за ними бачить, де починається й закінчується кожна репліка і хто говорить;
- **блок `## Metadata`** у системному повідомленні:
  - `Knowledge Cutoff Date: June 2025` - до якої дати модель знає світ;
  - `Today Date: 18 November 2026` - **сьогоднішня дата, підставлена автоматично** під час виклику (модель знає "сьогодні", хоча ми цього не передавали);
  - `Reasoning Mode: /no_think` або `/think` - прапорець, який ми дописали в кінець системного повідомлення, шаблон **вирізав з нашого тексту** й переніс сюди;
- **блок `## Custom Instructions`**: тут наш системний текст без прапорця (`You are a math tutor.`). Якщо ми передали лише прапорець, шаблон підставив
  **стандартну інструкцію**: `You are a helpful AI assistant named SmolLM, trained by Hugging Face.` Для `/think` без власного тексту (діалог `reasoning_task`)
  ця стандартна інструкція набагато довша: вона пояснює, що відповідь має складатися з розділів "Thought" у `<think>…</think>` і "Solution";
- у режимі `/no_think` в кінець промпту для інференсу (і перед попередньою відповіддю асистента в `multi_turn`) додається **порожній блок
  `<think>\n\n</think>`**. Так модель "бачить", що міркування вже завершені, і одразу пише відповідь. У режимі `/think` промпт закінчується
  просто на `<|im_start|>assistant`, і блок `<think>` модель відкриває й заповнює сама;
- у `multi_turn` попередня відповідь асистента лишилась в історії, а новий заголовок `assistant` додано лише після останнього повідомлення користувача.

Помітив ще одну деталь: системний блок SmolLM3 у цьому виводі закривається не `<|im_end|>`, а одразу переходить до `<|im_start|>user`.
Це особливість саме її шаблону. Ще одна причина ніколи не писати спецтокени вручну, а завжди використовувати `apply_chat_template`.

Навіщо все це: модель на етапі SFT тренували саме з таким системним блоком (дата, режим, інструкція), тож вона поводиться правильно,
лише коли бачить знайомий формат. Шаблон гарантує, що кожен запит виглядає як тренувальні дані, навіть якщо користувач передав лише одне питання.

**15. Прокоментуйте різницю між Base і Instruct, проглянувши вивід в секції 3.**

**Base-модель** (сирий текст "Explain me, what is quantum computing?" без шаблону) поводиться як **продовжувач інтернет-тексту**, а не як співрозмовник:
- почала писати статтю від першої особи: *"This is the question I've always wanted to know the answer to… In this article, I'll try to explain…"*;
- текст розмитий і порожній по суті, повторює одні й ті самі фрази та тавтології (*"Quantum information is a type of information that can be used to process information"*);
- не пояснила жодного реального поняття (кубіт, суперпозиція), не зупинилась сама - обірвалась на ліміті 150 токенів.

**Instruct-модель** (той самий запит у шаблоні розмови) **відповіла саме на питання**:
одразу дала визначення, порівняла біти з кубітами й навела структурований список із пунктами **Qubits**, **Superposition**, **Entanglement** з Markdown-розміткою.
Це відповідь асистента, а не уривок зі статті.

**Чому так:** обидві моделі мають однакову архітектуру й однакове попереднє тренування. Base вчилась лише передбачати наступний токен в інтернет-текстах,
тож продовжує текст найімовірнішим чином (часто статтею чи форумом). Instruct додатково донавчали (SFT + вирівнювання) на діалогах у шаблоні розмови,
і вона навчилась розуміти, що після `<|im_start|>assistant` треба відповісти на питання користувача. Саме тому для SFT зазвичай беруть instruct-модель,
або для base-моделі спочатку задають шаблон (секція "А що з base-моделлю?": у base-токенізаторі шаблону взагалі немає - `False`).

**16. Чому в режимі `/think` при генерації було передано параметр `max_new_tokens=300`?**

У режимі `/think` модель спочатку пише міркування в блоці `<think>…</think>` і лише потім відповідь. Міркування займають
десятки-сотні токенів, тож зі стандартними для функції `generate` 150 токенами генерація часто обривалася б **посеред міркувань**,
і ми взагалі не побачили б відповіді. 300 - компроміс: вистачає на міркування й відповідь для простих арифметичних задач,
але генерація не триває надто довго. Для режиму `/no_think` така межа теж не шкодить: модель зупиниться сама на `<|im_end|>`.

**17. Чи впоралася модель з математичними задачами? Який би режим генерації Ви обрали для себе, якщо б Вам потрібна була допомога?**

**Режим `/no_think` - усі три задачі розв'язано правильно:**

| задача | відповідь моделі | правильно |
|---|---|---|
| 15 × 24 | 360 (множення стовпчиком + перевірка через 15 × (20 + 4)) | ✅ |
| 2 чашки на 12 печив → на 30 | 5 cups (через 1/6 чашки на печиво) | ✅ |
| $50 − $18.75 − $12.30 | $18.95 | ✅ |

Відповіді покрокові й повні, кожна вклалась у ліміт.

**Режим `/think` - міркування правильні, але відповіді немає в жодній задачі.** Модель міркує дуже докладно
(*"Hmm, let me think… Wait, is that right? Let me check my steps again"*), і **300 токенів закінчились усередині блоку `<think>`**:
- у задачі 1 вона вже отримала 360, але почала перевіряти (*"Let me check if"*) і обірвалась;
- у задачі 2 отримала 5 cups і почала перевіряти ще раз;
- у задачі 3 порахувала $31.05 витрат і обірвалась на відніманні.

Розділу "Solution" з остаточною відповіддю так і не з'явилось. Користувач побачив би лише незавершені роздуми.

**Що обрав би я:** для простих задач, як ці, - **`/no_think`**. Він правильний, у кілька разів швидший і одразу дає зрозумілу відповідь з кроками.
`/think` вмикав би для складних багатокрокових задач (олімпіадна математика, логіка, код), де міркування справді підвищують точність, але тоді
з більшим `max_new_tokens` (1-2 тис.) і з готовністю довше чекати.

**18. Які відмінності в шаблонах розмов Ви помітили в секції 5?**

Один і той самий діалог (`system` + `user` + `assistant`) три моделі записують по-різному:

| | SmolLM3-3B | Qwen3-0.6B | Phi-3.5-mini-instruct |
|---|---|---|---|
| формат ролей | ChatML: `<|im_start|>system` … | ChatML: `<|im_start|>system` … `<|im_end|>` | свої токени: `<|system|>`, `<|user|>`, `<|assistant|>` |
| кінець репліки | `<|im_end|>` (у системного блоку немає) | `<|im_end|>` | `<|end|>` + `<|endoftext|>` у кінці діалогу |
| системне повідомлення | обгорнуте в `## Metadata` (дата відсічення, сьогоднішня дата, `Reasoning Mode: /think`) і `## Custom Instructions` | **як є**, без доповнень | як є |
| міркування | режим `/think` за замовчуванням (у метаданих), блоку в відповіді немає | у відповідь асистента вставлено **порожній `<think>\n\n</think>`** | режиму міркувань немає |

Режим міркувань теж вмикається по-різному: у SmolLM3 прапорцем `/think` / `/no_think` у системному повідомленні,
у Qwen3 параметром `enable_thinking` в `apply_chat_template`. В останній клітинці видно, що з `enable_thinking=True` промпт закінчується на
`<|im_start|>assistant`, а з `False` до нього додається порожній блок `<think></think>`.

**Висновок:** ролі `system` / `user` / `assistant` - це лише домовленість на рівні `messages`. У токенах кожна модель має свій формат, на якому її
тренували. Якщо подати Phi-3.5 діалог у форматі ChatML (чи навпаки), вона бачитиме незнайомі токени й працюватиме гірше. Тому шаблон завжди
беруть з токенізатора конкретної моделі, а не пишуть вручну.

**19. Для чого нам потрібно знати про шаблони розмов, якщо ми хочемо донавчити модель за допомогою SFT?**

- **Дані мають бути в тому самому форматі, що й під час використання.** Модель вивчає послідовність токенів; якщо тренувати з одним
  шаблоном, а запускати з іншим (чи без шаблону), вона бачить незнайомий формат і працює гірше.
- **Кожна модель має свій шаблон** (секція 5): готуємо дані в нейтральному форматі `messages`, а шаблон конкретної моделі
  застосовує токенізатор/TRL. У base-моделі шаблону може не бути - тоді його треба задати самим (взяти з instruct-версії).
- **Модель вчиться зупинятися**: токен кінця репліки (`<|im_end|>`) у тренувальних даних вчить модель завершувати відповідь.
- **Loss лише на відповідях**: щоб рахувати loss тільки на репліках `assistant` (`assistant_only_loss` у демо 3), потрібно знати, де в токенах вони починаються й закінчуються.
- **Режим міркування**: шаблон сам додає метадані й блок `<think>`. Як показано в кінці ноутбука, якщо дані без міркувань форматувати в режимі `/think`,
  модель розучиться міркувати. Тож режим у шаблоні має збігатися з тим, як модель використовуватиметься.

**20. З яких даних було побудовано датасет `HuggingFaceTB/smoltalk2`?**

**SmolTalk2** - збірка з **25 частин** (у SFT-конфігурації), з яких 9 зібрані для режиму `/think` (відповіді з міркуваннями `<think>…</think>`)
і 16 для `/no_think`. За назвами частин джерела такі:
- **синтетичні діалоги, згенеровані Qwen3-32B** (у назві `Qwen3_32B`): повсякденні розмови з міркуваннями (`smoltalk_everyday_convs_reasoning`), системні чати,
  багатомовні розмови (`smoltalk_multilingual8`, `aya_dataset`), таблиці (`table_gpt`), довгий контекст (`LongAlign_64k`);
- **готові відкриті датасети міркувань:** `OpenThoughts3_1.2M` (математика, код, наука), `s1k_1.1` (складні математичні задачі), `Mixture_of_Thoughts_science`;
- **виконання інструкцій:** `tulu_3_sft_personas_instruction_following`, `multi_turn_reasoning_if`, `OpenHermes_2.5`;
- **виклик інструментів і агенти:** `smolagents_toolcalling_traces`, `xlam_traces`, `hermes_function_calling_v1`;
- **частини першої версії SmolTalk**, перегенеровані для SmolLM3: повсякденні розмови, переписування й скорочення текстів (`smol_rewrite`, `smol_summarize`),
  `smol_magpie_ultra` (діалоги, згенеровані методом Magpie), `explore_instruct_rewriting`, `systemchats_30k`, багатомовні (8 мов).

Тобто датасет поєднує **відкриті людські й синтетичні датасети** та **синтетичні відповіді сильнішої моделі** (Qwen3-32B). Кожен приклад має колонки
`messages`, `chat_template_kwargs` (режим міркувань) і `source`. Завантажена для прикладу частина `smoltalk_everyday_convs_reasoning_Qwen3_32B_think`
містить 2 057 діалогів, і вже в першому видно міркування Qwen3-32B в `<think>` навіть на просте "Hi there".

**21. Які модифікації даних відбуваються в секції "Перетворення власного датасету у формат messages"?**

- з GSM8K береться лише перші **100** прикладів (`train[:100]`), щоб демо працювало швидко;
- кожна пара колонок `question` / `answer` перетворюється на **діалог із трьох повідомлень**:
  - `system` - **нова** інструкція, якої в датасеті не було: *"You are a math tutor. Solve problems step by step."* (однакова для всіх прикладів);
  - `user` - текст задачі (`question`) без змін;
  - `assistant` - розв'язок (`answer`) без змін: покрокові обчислення з анотаціями калькулятора `<<48/2=24>>` і фінальною відповіддю після `####`;
- оригінальні колонки видаляються (`remove_columns`), залишається одна колонка `messages`.

Сам текст задач і відповідей не змінюється - змінюється лише структура (з табличної в діалогову). Потім, у наступній клітинці,
шаблон розмови додає до цього діалогу ще метадані SmolLM3, спецтокени ролей і режим `/think`.